# Autoencoder

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
import torch
import numpy as np
import torch.optim as optim
import torch.nn.functional as F

sys.path.append('..')

from utils.train import train
from utils.scheduler import exponential_decay
from utils.data_processor import create_flower_dataloaders

In [8]:
# Set seeds
torch.manual_seed(0)
np.random.seed(0)

data_root = "../data/flowers"
model_save_root = "../model"

batch_size = 16
img_width, img_height = 24, 24

training_dataloader, validation_dataloader = create_flower_dataloaders(batch_size, data_root, img_width, img_height)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

## MLP Autoencoder

In [9]:
from MLPAE import MLPAutoencoder, MLPAE_ENCODING_DIM

num_epochs = 100
early_stopping_patience = 5
lr = 1.

model = MLPAutoencoder(MLPAE_ENCODING_DIM, img_width, img_height)
loss_fn = F.mse_loss

optimizer = optim.SGD(
    model.parameters(), 
    momentum=0., 
    lr=lr
) 
scheduler = exponential_decay(
    initial_learning_rate=lr, 
    decay_rate=0.9, 
    decay_epochs=10
) 

model.to(device)

# Start training
train(
    optimizer=optimizer,
    scheduler=scheduler,
    model=model,
    training_dataloader=training_dataloader,
    validation_dataloader=validation_dataloader,
    num_epochs=num_epochs,
    early_stopping_patience=early_stopping_patience,
    device=device,
    model_save_root=model_save_root,
    loss_fn=loss_fn,
)

Model saved at epoch 0, val_loss=0.078642
Epoch: 0, Train Loss: 0.08089493848383426, Valid Loss: 0.07864188583511295
Epoch: 1, Train Loss: 0.07995381833842168, Valid Loss: 0.07864188583511295
Epoch: 2, Train Loss: 0.07850466018112806, Valid Loss: 0.07864188583511295
Model saved at epoch 3, val_loss=0.059498
Epoch: 3, Train Loss: 0.07070017319459182, Valid Loss: 0.059497514005863304
Epoch: 4, Train Loss: 0.05274572968482971, Valid Loss: 0.059497514005863304
Epoch: 5, Train Loss: 0.04700525185236564, Valid Loss: 0.059497514005863304


KeyboardInterrupt: 

## CNN Autoencoder

In [10]:
from CNNAE import CNNAutoencoder, CNNAE_ENCODING_DIM

num_epochs = 30   
early_stopping_patience = 5
lr = 1e-3

model = CNNAutoencoder(CNNAE_ENCODING_DIM) 
loss_fn = F.mse_loss

optimizer = optim.Adam(
    model.parameters(),
    lr=lr,
    betas=(0.9, 0.999),
    weight_decay=1e-5
)
scheduler = exponential_decay(
    initial_learning_rate=lr,
    decay_rate=0.8,      
    decay_epochs=5       
)

model.to(device)

# Start training
train(
    optimizer=optimizer,
    scheduler=scheduler,
    model=model,
    training_dataloader=training_dataloader,
    validation_dataloader=validation_dataloader,
    num_epochs=num_epochs,
    early_stopping_patience=early_stopping_patience,
    device=device,
    model_save_root=model_save_root,
    loss_fn=loss_fn,
)

Model saved at epoch 0, val_loss=0.023743
Epoch: 0, Train Loss: 0.038487203161303815, Valid Loss: 0.023742970135627373


KeyboardInterrupt: 